In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

folder = Path("../output_csv")

input_file = folder / "residential_cleaned.csv"

df = pd.read_csv(input_file, low_memory=False)

df.head()

,ClosePrice,source_month,LivingArea,DaysOnMarket,LotSizeSquareFeet,YearBuilt,BathroomsTotalInteger,BedroomsTotal,GarageSpaces,Latitude,...,PostalCode_92584,PostalCode_92592,PostalCode_92596,PostalCode_93065,PostalCode_93446,PostalCode_93535,PostalCode_93536,PostalCode_93551,PostalCode_94513,PostalCode_Other
0,890000.0,202506,3000.0,181,9600.0,2021.0,3.0,3.0,2.0,34.264692,...,False,False,False,False,False,False,False,False,False,True
1,1876384.0,202506,1800.0,87,10400.0,1963.0,3.0,3.0,2.0,34.107983,...,False,False,False,False,False,False,False,False,False,True
2,4820000.0,202506,4270.0,0,22505.0,1980.0,6.0,6.0,3.0,37.567434,...,False,False,False,False,False,False,False,False,False,True
3,865000.0,202506,1442.0,0,4800.0,1985.0,2.0,3.0,2.0,33.906058,...,False,False,False,False,False,False,False,False,False,True
4,875000.0,202506,1086.0,0,5500.0,1953.0,1.0,3.0,4.0,37.705919,...,False,False,False,False,False,False,False,False,False,True


In [3]:
target = 'ClosePrice'
# Train/test split by most recent month
df['source_month'] = df['source_month'].astype(float).astype(int).astype(str)

months = sorted(df['source_month'].unique())

test_month = months[-1]

In [4]:
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

target = "ClosePrice"

df["source_month"] = (
    df["source_month"]
    .astype(float)
    .astype(int)
    .astype(str)
)

months = sorted(df["source_month"].unique())
test_month = months[-1]

drop_cols = [target, "source_month"]

feature_cols = [
    col for col in df.columns
    if col not in drop_cols
]

models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(
        random_state=42
    ),
    "Random Forest": RandomForestRegressor(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    )
}

model_results = []

for X_window in [3, 6, 9, 12]:

    train_months = months[-(X_window + 1):-1]

    train_df = df[
        df["source_month"].isin(train_months)
    ].copy()

    test_df = df[
        df["source_month"] == test_month
    ].copy()

    X_train = train_df[feature_cols]
    y_train = train_df[target]

    X_test = test_df[feature_cols]
    y_test = test_df[target]

    for model_name, estimator in models.items():

        # Fresh copy of each estimator for every training window
        model = clone(estimator)

        model.fit(X_train, y_train)

        train_preds = model.predict(X_train)
        test_preds = model.predict(X_test)

        model_results.append({
            "model": model_name,
            "training_window_months": X_window,
            "train_months": ", ".join(train_months),
            "test_month": test_month,
            "train_rows": len(train_df),
            "test_rows": len(test_df),
            "feature_count": len(feature_cols),
            "train_r2": r2_score(
                y_train,
                train_preds
            ),
            "test_r2": r2_score(
                y_test,
                test_preds
            ),
            "mae": mean_absolute_error(
                y_test,
                test_preds
            ),
            "rmse": np.sqrt(
                mean_squared_error(
                    y_test,
                    test_preds
                )
            )
        })

comparison_df = pd.DataFrame(model_results)

comparison_df = comparison_df.sort_values(
    by="test_r2",
    ascending=False
).reset_index(drop=True)

comparison_df

,model,training_window_months,train_months,test_month,train_rows,test_rows,feature_count,train_r2,test_r2,mae,rmse
0,Linear Regression,12,"202506, 202507, 202508, 202509, 202510, 202511...",202606,129591,12793,200,0.014958,0.472322,572191.011960,1.116176e+06
1,Linear Regression,6,"202512, 202601, 202602, 202603, 202604, 202605",202606,61429,12793,200,0.015931,0.468593,556753.222951,1.120113e+06
2,Linear Regression,3,"202603, 202604, 202605",202606,35089,12793,200,0.014623,0.468579,566923.592680,1.120128e+06
3,Linear Regression,9,"202509, 202510, 202511, 202512, 202601, 202602...",202606,94495,12793,200,0.009956,0.465947,560517.607724,1.122899e+06
4,Random Forest,12,"202506, 202507, 202508, 202509, 202510, 202511...",202606,129591,12793,200,0.841250,-6.433140,352407.493405,4.189231e+06
5,Random Forest,6,"202512, 202601, 202602, 202603, 202604, 202605",202606,61429,12793,200,0.826959,-9.470706,351271.454259,4.972057e+06
6,Random Forest,3,"202603, 202604, 202605",202606,35089,12793,200,0.843456,-9.525622,381037.588263,4.985079e+06
7,Random Forest,9,"202509, 202510, 202511, 202512, 202601, 202602...",202606,94495,12793,200,0.835601,-15.224120,435740.536669,6.189117e+06
8,Decision Tree,12,"202506, 202507, 202508, 202509, 202510, 202511...",202606,129591,12793,200,0.999988,-18.097696,364507.844717,6.714886e+06
9,Decision Tree,3,"202603, 202604, 202605",202606,35089,12793,200,1.000000,-32.807832,465317.242103,8.934223e+06


In [5]:
# Find the best Linear Regression baseline
best_baseline_row = (
    comparison_df[
        comparison_df["model"] == "Linear Regression"
    ]
    .sort_values(
        by="test_r2",
        ascending=False
    )
    .iloc[0]
)

best_baseline_r2 = best_baseline_row["test_r2"]
best_baseline_window = best_baseline_row["training_window_months"]

print("Best baseline window:", best_baseline_window)
print(f"Best baseline test R²: {best_baseline_r2:.4f}")

Best baseline window: 12
Best baseline test R²: 0.4723


In [6]:
comparison_df["baseline_test_r2"] = best_baseline_r2

comparison_df["r2_change_from_baseline"] = (
    comparison_df["test_r2"]
    - comparison_df["baseline_test_r2"]
)

comparison_df.sort_values(
    by="test_r2",
    ascending=False
)

,model,training_window_months,train_months,test_month,train_rows,test_rows,feature_count,train_r2,test_r2,mae,rmse,baseline_test_r2,r2_change_from_baseline
0,Linear Regression,12,"202506, 202507, 202508, 202509, 202510, 202511...",202606,129591,12793,200,0.014958,0.472322,572191.011960,1.116176e+06,0.472322,0.000000
1,Linear Regression,6,"202512, 202601, 202602, 202603, 202604, 202605",202606,61429,12793,200,0.015931,0.468593,556753.222951,1.120113e+06,0.472322,-0.003730
2,Linear Regression,3,"202603, 202604, 202605",202606,35089,12793,200,0.014623,0.468579,566923.592680,1.120128e+06,0.472322,-0.003743
3,Linear Regression,9,"202509, 202510, 202511, 202512, 202601, 202602...",202606,94495,12793,200,0.009956,0.465947,560517.607724,1.122899e+06,0.472322,-0.006376
4,Random Forest,12,"202506, 202507, 202508, 202509, 202510, 202511...",202606,129591,12793,200,0.841250,-6.433140,352407.493405,4.189231e+06,0.472322,-6.905463
5,Random Forest,6,"202512, 202601, 202602, 202603, 202604, 202605",202606,61429,12793,200,0.826959,-9.470706,351271.454259,4.972057e+06,0.472322,-9.943028
6,Random Forest,3,"202603, 202604, 202605",202606,35089,12793,200,0.843456,-9.525622,381037.588263,4.985079e+06,0.472322,-9.997944
7,Random Forest,9,"202509, 202510, 202511, 202512, 202601, 202602...",202606,94495,12793,200,0.835601,-15.224120,435740.536669,6.189117e+06,0.472322,-15.696442
8,Decision Tree,12,"202506, 202507, 202508, 202509, 202510, 202511...",202606,129591,12793,200,0.999988,-18.097696,364507.844717,6.714886e+06,0.472322,-18.570018
9,Decision Tree,3,"202603, 202604, 202605",202606,35089,12793,200,1.000000,-32.807832,465317.242103,8.934223e+06,0.472322,-33.280154


In [7]:
comparison_df["beat_baseline"] = (
    comparison_df["test_r2"] > best_baseline_r2
)

comparison_df[
    [
        "model",
        "training_window_months",
        "test_r2",
        "baseline_test_r2",
        "r2_change_from_baseline",
        "beat_baseline"
    ]
].sort_values(
    by="test_r2",
    ascending=False
)

,model,training_window_months,test_r2,baseline_test_r2,r2_change_from_baseline,beat_baseline
0,Linear Regression,12,0.472322,0.472322,0.000000,False
1,Linear Regression,6,0.468593,0.472322,-0.003730,False
2,Linear Regression,3,0.468579,0.472322,-0.003743,False
3,Linear Regression,9,0.465947,0.472322,-0.006376,False
4,Random Forest,12,-6.433140,0.472322,-6.905463,False
5,Random Forest,6,-9.470706,0.472322,-9.943028,False
6,Random Forest,3,-9.525622,0.472322,-9.997944,False
7,Random Forest,9,-15.224120,0.472322,-15.696442,False
8,Decision Tree,12,-18.097696,0.472322,-18.570018,False
9,Decision Tree,3,-32.807832,0.472322,-33.280154,False


# week 6

In [8]:
df_engineered = df.copy()

# Convert source month into year and month components
df_engineered["sale_year"] = (
    df_engineered["source_month"]
    .str[:4]
    .astype(int)
)

df_engineered["sale_month_number"] = (
    df_engineered["source_month"]
    .str[4:6]
    .astype(int)
)

df_engineered["sale_quarter"] = (
    (df_engineered["sale_month_number"] - 1) // 3 + 1
)

# Cyclical month features
df_engineered["month_sin"] = np.sin(
    2 * np.pi * df_engineered["sale_month_number"] / 12
)

df_engineered["month_cos"] = np.cos(
    2 * np.pi * df_engineered["sale_month_number"] / 12
)

In [9]:
# property age features


if "YearBuilt" in df_engineered.columns:

    df_engineered["property_age"] = (
        df_engineered["sale_year"]
        - df_engineered["YearBuilt"]
    )

    # Remove impossible negative ages
    df_engineered.loc[
        df_engineered["property_age"] < 0,
        "property_age"
    ] = np.nan

    df_engineered["property_age_squared"] = (
        df_engineered["property_age"] ** 2
    )

    df_engineered["is_new_property"] = (
        df_engineered["property_age"] <= 5
    ).astype(int)

    df_engineered["is_old_property"] = (
        df_engineered["property_age"] >= 50
    ).astype(int)

In [10]:
# bedroom and bathroom features 

if (
    "BedroomsTotal" in df_engineered.columns
    and "BathroomsTotalInteger" in df_engineered.columns
):

    bedrooms = df_engineered["BedroomsTotal"]
    bathrooms = df_engineered["BathroomsTotalInteger"]

    df_engineered["bed_bath_ratio"] = (
        bedrooms
        / bathrooms.replace(0, np.nan)
    )

    df_engineered["bath_bed_ratio"] = (
        bathrooms
        / bedrooms.replace(0, np.nan)
    )

    df_engineered["total_bed_bath"] = (
        bedrooms + bathrooms
    )

    df_engineered["bed_bath_difference"] = (
        bedrooms - bathrooms
    )

    df_engineered["more_bathrooms_than_bedrooms"] = (
        bathrooms > bedrooms
    ).astype(int)

In [11]:
# living area features 

if "LivingArea" in df_engineered.columns:

    df_engineered["living_area_squared"] = (
        df_engineered["LivingArea"] ** 2
    )

    df_engineered["log_living_area"] = np.log1p(
        df_engineered["LivingArea"].clip(lower=0)
    )

    df_engineered["small_home_flag"] = (
        df_engineered["LivingArea"] < 1000
    ).astype(int)

    df_engineered["large_home_flag"] = (
        df_engineered["LivingArea"] > 3000
    ).astype(int)


if (
    "LivingArea" in df_engineered.columns
    and "BedroomsTotal" in df_engineered.columns
):

    df_engineered["living_area_per_bedroom"] = (
        df_engineered["LivingArea"]
        / df_engineered["BedroomsTotal"].replace(0, np.nan)
    )


if (
    "LivingArea" in df_engineered.columns
    and "BathroomsTotalInteger" in df_engineered.columns
):

    df_engineered["living_area_per_bathroom"] = (
        df_engineered["LivingArea"]
        / df_engineered[
            "BathroomsTotalInteger"
        ].replace(0, np.nan)
    )

In [12]:
# lot size features

if "LotSizeSquareFeet" in df_engineered.columns:

    df_engineered["log_lot_size"] = np.log1p(
        df_engineered[
            "LotSizeSquareFeet"
        ].clip(lower=0)
    )

    df_engineered["large_lot_flag"] = (
        df_engineered["LotSizeSquareFeet"] > 10000
    ).astype(int)

    df_engineered["small_lot_flag"] = (
        df_engineered["LotSizeSquareFeet"] < 3000
    ).astype(int)



if (
    "LotSizeSquareFeet" in df_engineered.columns
    and "LivingArea" in df_engineered.columns
):

    df_engineered["lot_to_living_ratio"] = (
        df_engineered["LotSizeSquareFeet"]
        / df_engineered["LivingArea"].replace(0, np.nan)
    )

    df_engineered["building_coverage_ratio"] = (
        df_engineered["LivingArea"]
        / df_engineered[
            "LotSizeSquareFeet"
        ].replace(0, np.nan)
    )

    df_engineered["unused_lot_area"] = (
        df_engineered["LotSizeSquareFeet"]
        - df_engineered["LivingArea"]
    )

In [13]:
# story features 

if "Stories" in df_engineered.columns:

    df_engineered["single_story_flag"] = (
        df_engineered["Stories"] == 1
    ).astype(int)

    df_engineered["multi_story_flag"] = (
        df_engineered["Stories"] > 1
    ).astype(int)


if "Stories" in df_engineered.columns:

    df_engineered["single_story_flag"] = (
        df_engineered["Stories"] == 1
    ).astype(int)

    df_engineered["multi_story_flag"] = (
        df_engineered["Stories"] > 1
    ).astype(int)

In [14]:
missing_indicator_columns = [
    "LivingArea",
    "LotSizeSquareFeet",
    "YearBuilt",
    "BedroomsTotal",
    "BathroomsTotalInteger",
    "AssociationFee",
    "Stories",
    "MainLevelBedrooms"
]

for column in missing_indicator_columns:

    if column in df_engineered.columns:

        df_engineered[
            f"{column}_missing"
        ] = (
            df_engineered[column]
            .isna()
            .astype(int)
        )

df_engineered["missing_feature_count"] = (
    df_engineered.isna().sum(axis=1)
)


# clean infinite values

df_engineered = df_engineered.replace(
    [np.inf, -np.inf],
    np.nan
)

In [15]:
# fill with medians 

numeric_columns = (
    df_engineered
    .select_dtypes(include=np.number)
    .columns
)

df_engineered[numeric_columns] = (
    df_engineered[numeric_columns]
    .fillna(
        df_engineered[numeric_columns].median()
    )
)

In [16]:
import geopandas as gpd


folder = Path("../output_csv")


school_district_file = (
    folder / "DistrictAreas2526_-284845464123469011.geojson"
)

school_districts = gpd.read_file(
    school_district_file
)

print("School district shape:", school_districts.shape)
print("Original CRS:", school_districts.crs)
print("Columns:", school_districts.columns.tolist())


district_name_column = "DistrictName"
district_type_column = "DistrictType"


# --------------------------------------------------
# 2. Inspect the actual DistrictType values
# --------------------------------------------------

print(
    school_districts[district_type_column]
    .value_counts(dropna=False)
)


# Clean the district type text
school_districts[district_type_column] = (
    school_districts[district_type_column]
    .astype("string")
    .str.strip()
)


# Keep only Unified School Districts.
# contains() is safer than == "Unified" because the value
# could be something like "Unified School District".
unified_districts = school_districts[
    school_districts[district_type_column]
    .str.contains(
        "Unified",
        case=False,
        na=False
    )
].copy()

print(
    "Unified district polygons:",
    len(unified_districts)
)

print(
    unified_districts[
        [
            district_name_column,
            district_type_column
        ]
    ].head()
)


if unified_districts.empty:
    raise ValueError(
        "No Unified districts were found. "
        "Review the DistrictType values printed above."
    )


# --------------------------------------------------
# 3. Standardize the district CRS
# --------------------------------------------------

# GeoJSON should normally use EPSG:4326, but this ensures
# that the polygons and property coordinates use the same CRS.
if unified_districts.crs is None:
    unified_districts = unified_districts.set_crs(
        "EPSG:4326"
    )
else:
    unified_districts = unified_districts.to_crs(
        "EPSG:4326"
    )


# Fix invalid polygon geometries if any exist
unified_districts["geometry"] = (
    unified_districts.geometry.make_valid()
)

unified_districts = unified_districts[
    unified_districts.geometry.notna()
    & ~unified_districts.geometry.is_empty
].copy()


# Only retain fields needed for the spatial join
district_layer = unified_districts[
    [
        district_name_column,
        district_type_column,
        "geometry"
    ]
].copy()


# --------------------------------------------------
# 4. Clean property coordinates
# --------------------------------------------------

df_engineered = df_engineered.copy()

# Preserve each property's original position
df_engineered["_row_id"] = np.arange(
    len(df_engineered)
)


# Convert coordinates from text to numeric values
df_engineered["Latitude"] = pd.to_numeric(
    df_engineered["Latitude"],
    errors="coerce"
)

df_engineered["Longitude"] = pd.to_numeric(
    df_engineered["Longitude"],
    errors="coerce"
)


# California's approximate coordinate range.
# This also catches accidentally reversed coordinates.
valid_coordinate_mask = (
    df_engineered["Latitude"].between(
        32,
        43
    )
    & df_engineered["Longitude"].between(
        -125,
        -113
    )
)

print(
    "Properties with valid California coordinates:",
    valid_coordinate_mask.sum()
)

print(
    "Properties with invalid or missing coordinates:",
    (~valid_coordinate_mask).sum()
)


# Helpful diagnostic
print(
    df_engineered.loc[
        valid_coordinate_mask,
        ["Latitude", "Longitude"]
    ].head()
)


properties_with_coordinates = (
    df_engineered.loc[
        valid_coordinate_mask
    ]
    .copy()
)

properties_without_coordinates = (
    df_engineered.loc[
        ~valid_coordinate_mask
    ]
    .copy()
)


# --------------------------------------------------
# 5. Convert properties into geographic points
# --------------------------------------------------

properties_gdf = gpd.GeoDataFrame(
    properties_with_coordinates,
    geometry=gpd.points_from_xy(
        properties_with_coordinates["Longitude"],
        properties_with_coordinates["Latitude"]
    ),
    crs="EPSG:4326"
)


# Both layers should now use EPSG:4326
print("Property CRS:", properties_gdf.crs)
print("District CRS:", district_layer.crs)


# --------------------------------------------------
# 6. Spatially join properties to Unified districts
# --------------------------------------------------

properties_joined = gpd.sjoin(
    properties_gdf,
    district_layer,
    how="left",
    predicate="intersects"
)


# A point should generally match only one Unified district.
# This prevents duplicate property rows if district polygons overlap.
properties_joined = (
    properties_joined
    .sort_values("_row_id")
    .drop_duplicates(
        subset="_row_id",
        keep="first"
    )
)


# Remove GeoPandas-only fields
properties_joined = properties_joined.drop(
    columns=[
        "geometry",
        "index_right"
    ],
    errors="ignore"
)


# --------------------------------------------------
# 7. Add missing district fields to invalid coordinates
# --------------------------------------------------

properties_without_coordinates[
    district_name_column
] = pd.NA

properties_without_coordinates[
    district_type_column
] = pd.NA


# --------------------------------------------------
# 8. Recombine and restore original order
# --------------------------------------------------

df_engineered = pd.concat(
    [
        properties_joined,
        properties_without_coordinates
    ],
    ignore_index=True
)

df_engineered = (
    df_engineered
    .sort_values("_row_id")
    .reset_index(drop=True)
)


df_engineered[
    "school_district_missing"
] = (
    df_engineered[district_name_column]
    .isna()
    .astype(int)
)


school_district_match_rate = (
    df_engineered[district_name_column]
    .notna()
    .mean()
)

valid_coordinate_match_rate = (
    df_engineered.loc[
        valid_coordinate_mask,
        district_name_column
    ]
    .notna()
    .mean()
)


print(
    f"Overall school district match rate: "
    f"{school_district_match_rate:.2%}"
)

print(
    f"Match rate among valid coordinates: "
    f"{valid_coordinate_match_rate:.2%}"
)

print(
    df_engineered[
        [
            "Latitude",
            "Longitude",
            district_name_column,
            district_type_column,
            "school_district_missing"
        ]
    ].head()
)


# Remove temporary identifier if it is no longer needed
df_engineered = df_engineered.drop(
    columns="_row_id"
)


# --------------------------------------------------
# 9. Save the enriched dataset
# --------------------------------------------------

output_file = (
    folder / "residential_with_school_districts.csv"
)

df_engineered.to_csv(
    output_file,
    index=False
)



School district shape: (936, 51)
Original CRS: EPSG:3857
Columns: ['OBJECTID', 'Year', 'FedID', 'CDCode', 'CDSCode', 'CountyName', 'DistrictName', 'DistrictType', 'GradeLow', 'GradeHigh', 'GradeLowCensus', 'GradeHighCensus', 'AssistStatus', 'UpdateNotes', 'EnrollTotal', 'EnrollCharter', 'EnrollNonCharter', 'AAcount', 'AApct', 'AIcount', 'AIpct', 'AScount', 'ASpct', 'FIcount', 'FIpct', 'HIcount', 'HIpct', 'PIcount', 'PIpct', 'WHcount', 'WHpct', 'MRcount', 'MRpct', 'NRcount', 'NRpct', 'ELcount', 'ELpct', 'FOScount', 'FOSpct', 'HOMcount', 'HOMpct', 'MIGcount', 'MIGpct', 'SWDcount', 'SWDpct', 'SEDcount', 'SEDpct', 'DistrctAreaSqMi', 'LocaleCode', 'LocaleDesc', 'geometry']
DistrictType
Elementary    515
Unified       345
High           76
Name: count, dtype: int64
Unified district polygons: 345
            DistrictName DistrictType
0        Alameda Unified      Unified
1    Albany City Unified      Unified
2       Berkeley Unified      Unified
3  Castro Valley Unified      Unified
4        

C:\Users\luoxu\AppData\Local\Temp\ipykernel_23996\948470939.py:255: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_engineered = pd.concat(


Overall school district match rate: 75.77%
Match rate among valid coordinates: 75.79%
    Latitude   Longitude                   DistrictName DistrictType  \
0  34.264692 -117.221040       Rim of the World Unified      Unified   
1  34.107983 -118.390320            Los Angeles Unified      Unified   
2  37.567434 -122.388226                            NaN         <NA>   
3  33.906058 -117.777782  Placentia-Yorba Linda Unified      Unified   
4  37.705919 -122.059421          Castro Valley Unified      Unified   

   school_district_missing  
0                        0  
1                        0  
2                        1  
3                        0  
4                        0  


In [17]:
# numeric for district 
district_dummies = pd.get_dummies(
    df_engineered[
        [
            district_name_column,
            district_type_column
        ]
    ],
    prefix=[
        "school_district",
        "school_district_type"
    ],
    drop_first=True,
    dtype=int
)

df_engineered = pd.concat(
    [
        df_engineered.drop(
            columns=[
                district_name_column,
                district_type_column
            ]
        ),
        district_dummies
    ],
    axis=1
)

non_numeric_columns = (
    df_engineered
    .select_dtypes(
        exclude=np.number
    )
    .columns
    .tolist()
)

print("Non-numeric columns:", non_numeric_columns)

protected_columns = [
    target,
    "source_month"
]

remaining_text_columns = [
    column
    for column in non_numeric_columns
    if column not in protected_columns
]

df_engineered = df_engineered.drop(
    columns=remaining_text_columns,
    errors="ignore"
)

Non-numeric columns: ['source_month', 'City_Corona', 'City_Escondido', 'City_Fontana', 'City_Hemet', 'City_Hesperia', 'City_Huntington Beach', 'City_Indio', 'City_La Quinta', 'City_Lancaster', 'City_Long Beach', 'City_Los Angeles', 'City_Menifee', 'City_Moreno Valley', 'City_Murrieta', 'City_Oakland', 'City_Oceanside', 'City_Other', 'City_Palm Desert', 'City_Palmdale', 'City_Riverside', 'City_San Bernardino', 'City_San Diego', 'City_San Jose', 'City_Temecula', 'City_Victorville', 'CountyOrParish_Butte', 'CountyOrParish_Contra Costa', 'CountyOrParish_Fresno', 'CountyOrParish_Kern', 'CountyOrParish_Lake', 'CountyOrParish_Los Angeles', 'CountyOrParish_Madera', 'CountyOrParish_Merced', 'CountyOrParish_Monterey', 'CountyOrParish_Orange', 'CountyOrParish_Other', 'CountyOrParish_Riverside', 'CountyOrParish_Sacramento', 'CountyOrParish_San Benito', 'CountyOrParish_San Bernardino', 'CountyOrParish_San Diego', 'CountyOrParish_San Joaquin', 'CountyOrParish_San Luis Obispo', 'CountyOrParish_San Ma

In [18]:
df_engineered

,ClosePrice,source_month,LivingArea,DaysOnMarket,LotSizeSquareFeet,YearBuilt,BathroomsTotalInteger,BedroomsTotal,GarageSpaces,Latitude,...,school_district_Williams Unified,school_district_Willits Unified,school_district_Willows Unified,school_district_Windsor Unified,school_district_Wiseburn Unified,school_district_Woodlake Unified,school_district_Woodland Joint Unified,school_district_Yosemite Unified,school_district_Yuba City Unified,school_district_Yucaipa-Calimesa Joint Unified
0,890000.0,202506,3000.0,181,9600.0,2021.0,3.0,3.0,2.0,34.264692,...,0,0,0,0,0,0,0,0,0,0
1,1876384.0,202506,1800.0,87,10400.0,1963.0,3.0,3.0,2.0,34.107983,...,0,0,0,0,0,0,0,0,0,0
2,4820000.0,202506,4270.0,0,22505.0,1980.0,6.0,6.0,3.0,37.567434,...,0,0,0,0,0,0,0,0,0,0
3,865000.0,202506,1442.0,0,4800.0,1985.0,2.0,3.0,2.0,33.906058,...,0,0,0,0,0,0,0,0,0,0
4,875000.0,202506,1086.0,0,5500.0,1953.0,1.0,3.0,4.0,37.705919,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
142379,40000.0,202606,1274.0,597,5100.0,1972.0,1.0,2.0,0.0,34.820415,...,0,0,0,0,0,0,0,0,0,0
142380,865000.0,202606,2166.0,674,868586.4,1975.0,3.0,5.0,2.0,34.242304,...,0,0,0,0,0,0,0,0,0,0
142381,280000.0,202606,2034.0,704,11433.0,2025.0,2.0,3.0,2.0,37.846557,...,0,0,0,0,0,0,0,0,0,0
142382,595000.0,202606,2502.0,961,4389.0,1890.0,3.0,4.0,1.0,37.824500,...,0,0,0,0,0,0,0,0,0,0


In [25]:
output_file = (
    folder / "engineered.csv"
)

df_engineered.to_csv(
    output_file,
    index=False
)

In [24]:
df_engineered.columns.tolist()

['ClosePrice',
 'source_month',
 'LivingArea',
 'DaysOnMarket',
 'LotSizeSquareFeet',
 'YearBuilt',
 'BathroomsTotalInteger',
 'BedroomsTotal',
 'GarageSpaces',
 'Latitude',
 'Longitude',
 'AssociationFee',
 'Stories',
 'MainLevelBedrooms',
 'PoolPrivateYN',
 'FireplaceYN',
 'NewConstructionYN',
 'AttachedGarageYN',
 'ViewYN',
 'LivingArea_missing_flag',
 'DaysOnMarket_missing_flag',
 'LotSizeSquareFeet_missing_flag',
 'YearBuilt_missing_flag',
 'BathroomsTotalInteger_missing_flag',
 'BedroomsTotal_missing_flag',
 'GarageSpaces_missing_flag',
 'Latitude_missing_flag',
 'Longitude_missing_flag',
 'AssociationFee_missing_flag',
 'Stories_missing_flag',
 'MainLevelBedrooms_missing_flag',
 'PoolPrivateYN_missing_flag',
 'FireplaceYN_missing_flag',
 'NewConstructionYN_missing_flag',
 'AttachedGarageYN_missing_flag',
 'ViewYN_missing_flag',
 'sale_year',
 'sale_month_number',
 'sale_quarter',
 'month_sin',
 'month_cos',
 'property_age',
 'property_age_squared',
 'is_new_property',
 'is_old_p

In [19]:
# new vs old feature count

drop_cols = [
    target,
    "source_month"
]

old_feature_cols = [
    column
    for column in df.columns
    if column not in drop_cols
]

new_feature_cols = [
    column
    for column in df_engineered.columns
    if column not in drop_cols
]

print(
    "Old feature count:",
    len(old_feature_cols)
)

print(
    "New feature count:",
    len(new_feature_cols)
)

Old feature count: 200
New feature count: 392


In [20]:
models = {
    "Linear Regression": LinearRegression(),

    "Decision Tree": DecisionTreeRegressor(
        random_state=42
    ),

    "Random Forest": RandomForestRegressor(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    )
}

In [21]:
def evaluate_models(
    data,
    feature_cols,
    feature_set_name
):

    model_results = []

    months = sorted(
        data["source_month"].unique()
    )

    for X_window in [3, 6, 9, 12]:

        train_months = months[
            -(X_window + 1):-1
        ]

        test_month = months[-1]

        train_df = data[
            data["source_month"].isin(
                train_months
            )
        ].copy()

        test_df = data[
            data["source_month"]
            == test_month
        ].copy()

        X_train = train_df[feature_cols]
        y_train = train_df[target]

        X_test = test_df[feature_cols]
        y_test = test_df[target]

        for model_name, model in models.items():

            model.fit(
                X_train,
                y_train
            )

            train_preds = model.predict(
                X_train
            )

            test_preds = model.predict(
                X_test
            )

            train_r2 = r2_score(
                y_train,
                train_preds
            )

            test_r2 = r2_score(
                y_test,
                test_preds
            )

            mae = mean_absolute_error(
                y_test,
                test_preds
            )

            rmse = np.sqrt(
                mean_squared_error(
                    y_test,
                    test_preds
                )
            )

            model_results.append({
                "feature_set": feature_set_name,
                "model": model_name,
                "training_window_months": X_window,
                "train_months": ", ".join(
                    train_months
                ),
                "test_month": test_month,
                "feature_count": len(
                    feature_cols
                ),
                "train_r2": train_r2,
                "test_r2": test_r2,
                "mae": mae,
                "rmse": rmse
            })

    return pd.DataFrame(
        model_results
    )

In [22]:
old_results = evaluate_models(
    data=df,
    feature_cols=old_feature_cols,
    feature_set_name="Old feature set"
)

old_results.sort_values(
    by="test_r2",
    ascending=False
)

KeyboardInterrupt: 

In [ ]:
new_results = evaluate_models(
    data=df_engineered,
    feature_cols=new_feature_cols,
    feature_set_name=(
        "Engineered features "
        "+ school district"
    )
)

new_results.sort_values(
    by="test_r2",
    ascending=False
)

,feature_set,model,training_window_months,train_months,test_month,feature_count,train_r2,test_r2,mae,rmse
9,Engineered features + school district,Linear Regression,12,"202506, 202507, 202508, 202509, 202510, 202511...",202606,392,0.016734,0.477487,558305.167281,1.110699e+06
6,Engineered features + school district,Linear Regression,9,"202509, 202510, 202511, 202512, 202601, 202602...",202606,392,0.011350,0.457700,562460.188444,1.131535e+06
3,Engineered features + school district,Linear Regression,6,"202512, 202601, 202602, 202603, 202604, 202605",202606,392,0.018927,0.448763,556826.786945,1.140820e+06
0,Engineered features + school district,Linear Regression,3,"202603, 202604, 202605",202606,392,0.022170,0.325517,574425.397679,1.261925e+06
2,Engineered features + school district,Random Forest,3,"202603, 202604, 202605",202606,392,0.834906,-4.913348,343151.898168,3.736499e+06
11,Engineered features + school district,Random Forest,12,"202506, 202507, 202508, 202509, 202510, 202511...",202606,392,0.838875,-4.972378,310260.767652,3.755102e+06
5,Engineered features + school district,Random Forest,6,"202512, 202601, 202602, 202603, 202604, 202605",202606,392,0.818118,-6.036501,308488.462793,4.075928e+06
8,Engineered features + school district,Random Forest,9,"202509, 202510, 202511, 202512, 202601, 202602...",202606,392,0.834901,-6.243786,331506.364399,4.135528e+06
4,Engineered features + school district,Decision Tree,6,"202512, 202601, 202602, 202603, 202604, 202605",202606,392,1.000000,-13.225589,384266.590460,5.795398e+06
1,Engineered features + school district,Decision Tree,3,"202603, 202604, 202605",202606,392,1.000000,-13.560441,406046.222705,5.863209e+06


In [ ]:
feature_comparison_df = pd.concat(
    [
        old_results,
        new_results
    ],
    ignore_index=True
)

feature_comparison_df.sort_values(
    by="test_r2",
    ascending=False
)

,feature_set,model,training_window_months,train_months,test_month,feature_count,train_r2,test_r2,mae,rmse
21,Engineered features + school district,Linear Regression,12,"202506, 202507, 202508, 202509, 202510, 202511...",202606,392,0.016734,0.477487,558305.167281,1.110699e+06
9,Old feature set,Linear Regression,12,"202506, 202507, 202508, 202509, 202510, 202511...",202606,200,0.014958,0.472322,572191.011960,1.116176e+06
3,Old feature set,Linear Regression,6,"202512, 202601, 202602, 202603, 202604, 202605",202606,200,0.015931,0.468593,556753.222951,1.120113e+06
0,Old feature set,Linear Regression,3,"202603, 202604, 202605",202606,200,0.014623,0.468579,566923.592680,1.120128e+06
6,Old feature set,Linear Regression,9,"202509, 202510, 202511, 202512, 202601, 202602...",202606,200,0.009956,0.465947,560517.607724,1.122899e+06
18,Engineered features + school district,Linear Regression,9,"202509, 202510, 202511, 202512, 202601, 202602...",202606,392,0.011350,0.457700,562460.188444,1.131535e+06
15,Engineered features + school district,Linear Regression,6,"202512, 202601, 202602, 202603, 202604, 202605",202606,392,0.018927,0.448763,556826.786945,1.140820e+06
12,Engineered features + school district,Linear Regression,3,"202603, 202604, 202605",202606,392,0.022170,0.325517,574425.397679,1.261925e+06
14,Engineered features + school district,Random Forest,3,"202603, 202604, 202605",202606,392,0.834906,-4.913348,343151.898168,3.736499e+06
23,Engineered features + school district,Random Forest,12,"202506, 202507, 202508, 202509, 202510, 202511...",202606,392,0.838875,-4.972378,310260.767652,3.755102e+06


In [ ]:
feature_comparison_df = pd.concat(
    [
        old_results,
        new_results
    ],
    ignore_index=True
)

feature_comparison_df.sort_values(
    by="test_r2",
    ascending=False
)

,feature_set,model,training_window_months,train_months,test_month,feature_count,train_r2,test_r2,mae,rmse
21,Engineered features + school district,Linear Regression,12,"202506, 202507, 202508, 202509, 202510, 202511...",202606,392,0.016734,0.477487,558305.167281,1.110699e+06
9,Old feature set,Linear Regression,12,"202506, 202507, 202508, 202509, 202510, 202511...",202606,200,0.014958,0.472322,572191.011960,1.116176e+06
3,Old feature set,Linear Regression,6,"202512, 202601, 202602, 202603, 202604, 202605",202606,200,0.015931,0.468593,556753.222951,1.120113e+06
0,Old feature set,Linear Regression,3,"202603, 202604, 202605",202606,200,0.014623,0.468579,566923.592680,1.120128e+06
6,Old feature set,Linear Regression,9,"202509, 202510, 202511, 202512, 202601, 202602...",202606,200,0.009956,0.465947,560517.607724,1.122899e+06
18,Engineered features + school district,Linear Regression,9,"202509, 202510, 202511, 202512, 202601, 202602...",202606,392,0.011350,0.457700,562460.188444,1.131535e+06
15,Engineered features + school district,Linear Regression,6,"202512, 202601, 202602, 202603, 202604, 202605",202606,392,0.018927,0.448763,556826.786945,1.140820e+06
12,Engineered features + school district,Linear Regression,3,"202603, 202604, 202605",202606,392,0.022170,0.325517,574425.397679,1.261925e+06
14,Engineered features + school district,Random Forest,3,"202603, 202604, 202605",202606,392,0.834906,-4.913348,343151.898168,3.736499e+06
23,Engineered features + school district,Random Forest,12,"202506, 202507, 202508, 202509, 202510, 202511...",202606,392,0.838875,-4.972378,310260.767652,3.755102e+06


In [ ]:
r2_comparison_table = (
    feature_comparison_df
    .pivot_table(
        index=[
            "model",
            "training_window_months"
        ],
        columns="feature_set",
        values="test_r2"
    )
    .reset_index()
)

r2_comparison_table.columns.name = None

r2_comparison_table

,model,training_window_months,Engineered features + school district,Old feature set
0,Decision Tree,3,-13.560441,-32.807832
1,Decision Tree,6,-13.225589,-53.937929
2,Decision Tree,9,-55.977270,-37.532127
3,Decision Tree,12,-25.392849,-18.097696
4,Linear Regression,3,0.325517,0.468579
5,Linear Regression,6,0.448763,0.468593
6,Linear Regression,9,0.457700,0.465947
7,Linear Regression,12,0.477487,0.472322
8,Random Forest,3,-4.913348,-9.525622
9,Random Forest,6,-6.036501,-9.470706


In [ ]:
r2_comparison_table = (
    feature_comparison_df
    .pivot_table(
        index=[
            "model",
            "training_window_months"
        ],
        columns="feature_set",
        values="test_r2"
    )
    .reset_index()
)

r2_comparison_table.columns.name = None

r2_comparison_table

,model,training_window_months,Engineered features + school district,Old feature set
0,Decision Tree,3,-13.560441,-32.807832
1,Decision Tree,6,-13.225589,-53.937929
2,Decision Tree,9,-55.977270,-37.532127
3,Decision Tree,12,-25.392849,-18.097696
4,Linear Regression,3,0.325517,0.468579
5,Linear Regression,6,0.448763,0.468593
6,Linear Regression,9,0.457700,0.465947
7,Linear Regression,12,0.477487,0.472322
8,Random Forest,3,-4.913348,-9.525622
9,Random Forest,6,-6.036501,-9.470706


In [ ]:
best_baseline_row = (
    old_results[
        old_results["model"]
        == "Linear Regression"
    ]
    .sort_values(
        by="test_r2",
        ascending=False
    )
    .iloc[0]
)

best_baseline_r2 = (
    best_baseline_row["test_r2"]
)

best_baseline_window = (
    best_baseline_row[
        "training_window_months"
    ]
)

print(
    "Best baseline window:",
    best_baseline_window
)

print(
    f"Best baseline test R²: "
    f"{best_baseline_r2:.4f}"
)

Best baseline window: 12
Best baseline test R²: 0.4723


In [ ]:
# Sort models by highest test R²
comparison_df = comparison_df.sort_values(
    by="test_r2",
    ascending=False
).reset_index(drop=True)

# Save all model comparison results
output_file = folder / "model_comparison_results.csv"

comparison_df.to_csv(
    output_file,
    index=False
)

# Select the best overall model
best_model = comparison_df.iloc[0]

print("Best Model:", best_model["model"])
print(
    "Best training window:",
    best_model["training_window_months"],
    "months"
)
print("Training months:", best_model["train_months"])
print("Test month:", best_model["test_month"])
print(f"Training R²: {best_model['train_r2']:.4f}")
print(f"Test R²: {best_model['test_r2']:.4f}")
print(f"Test MAE: ${best_model['mae']:,.2f}")
print(f"Test RMSE: ${best_model['rmse']:,.2f}")

best_model_df = comparison_df.iloc[[0]]

best_model_file = folder / "best_model_result.csv"

best_model_df.to_csv(
    best_model_file,
    index=False
)

Best Model: Linear Regression
Best training window: 12 months
Training months: 202506, 202507, 202508, 202509, 202510, 202511, 202512, 202601, 202602, 202603, 202604, 202605
Test month: 202606
Training R²: 0.0150
Test R²: 0.4723
Test MAE: $572,191.01
Test RMSE: $1,116,175.51
